# 🔐 PRIVIA — Sistema Multi-Agente de Auditoría de Privacidad

Arquitectura **Multi-LLM** con **Redis como Vector Store** para auditoría normativa de privacidad.

---

## 📚 Base de conocimiento indexada en Redis
| Documento | Tipo |
|---|---|
| Ley 21.719 — Protección y Tratamiento de Datos Personales (Chile, 13-DIC-2024) | Normativa legal |
| NIST Privacy Framework 1.1 (CSWP 40, April 2025) | Framework internacional |
| Política de Protección de Datos Personales — Banco General | Política interna |

## 🏗️ Arquitectura del sistema
```
INPUT
 ↓
[PASO 0] Input Sanitizer      → Detecta y elimina PII real (RUT, email, IP)
 ↓
[PASO 2] Orquestador LLM      → Clasifica consulta → {type, tools_to_invoke}
 ↓              ↘
[PASO 3] Workers MCP        [INCOMPLETE] → Solicitar antecedentes
 ├── tool_search_normativa  → Redis RAG KNN (similitud coseno)
 └── tool_query_catalog     → Data Catalog SQL simulado
 ↓
[PASO 4] LLM Auditor          → Genera reporte preliminar con evidencia
 ↓
[PASO 5] Agente Fiscalizador  → Valida citas, PII, consistencia
 ↓
[PASO 6] Logger + Respuesta   → audit_status: OK | ISSUES | CORRECTED
```

## ⚙️ Especificaciones Redis RAG
| Parámetro | Valor |
|---|---|
| Modelo de embeddings | `text-embedding-3-small` (OpenAI) — 1536 dims |
| Chunking | Por artículo/sección — 800-1200 tokens + overlap 100-200 |
| Índice | `normativa_interna` |
| Clave documento | `normativa:doc:<md5>` → JSON metadata |
| Clave embedding | `normativa:emb:<md5>` → bytes float32 |
| Clave índice | `normativa:index` → SET de doc_ids |
| Metadata | document_name, document_type, section, chunk_id, criticality, topic, source_owner, updated_at |

## 📂 CELDA 0 — Setup del corpus normativo

Resuelve la ubicación del corpus (`corpus/*.json`) según el entorno:

- **Colab**: usa `/content/corpus/`. Si no existe, te pide subir los 3 JSON (Ley 21.719, NIST CSWP 40, Política Banco General).
- **Local**: asume `corpus/` junto al notebook.

El path final se exporta como variable de entorno `PRIVIA_CORPUS_DIR`, que la Celda 3 lee al cargar el corpus.


In [27]:
import os
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules


def _ensure_folder(name: str, expected_glob: str, file_descripcion: str) -> Path:
    """Resuelve <folder>/ (Colab → /content/<folder>/, local → ./<folder>/).
    Si el folder está vacío en Colab, pide subir los archivos."""
    if IS_COLAB:
        folder = Path(f'/content/{name}')
        folder.mkdir(parents=True, exist_ok=True)
        if not list(folder.glob(expected_glob)):
            print(f'📥 No se encontró {file_descripcion} en {folder}/.')
            print(f'   Sube los archivos ({expected_glob}):')
            from google.colab import files
            uploaded = files.upload()
            for fname, data in uploaded.items():
                (folder / fname).write_bytes(data)
    else:
        folder = Path.cwd() / name
    return folder


CORPUS_DIR  = _ensure_folder('corpus',  '*.json', 'corpus normativo (3 JSON)')
PROMPTS_DIR = _ensure_folder('prompts', '*.md',   'prompts (4 .md: orquestador, auditor, fiscalizador, query_expansion)')

os.environ['PRIVIA_CORPUS_DIR']  = str(CORPUS_DIR)
os.environ['PRIVIA_PROMPTS_DIR'] = str(PROMPTS_DIR)

corpus_files  = sorted(p.name for p in CORPUS_DIR.glob('*.json'))
prompt_files  = sorted(p.name for p in PROMPTS_DIR.glob('*.md'))
print(f'✅ Corpus dir : {CORPUS_DIR}  ({len(corpus_files)} JSON)')
print(f'✅ Prompts dir: {PROMPTS_DIR}  ({len(prompt_files)} MD)')
print(f'   corpus → {corpus_files}')
print(f'   prompts → {prompt_files}')


✅ Corpus dir : /Users/julissa.rodriguez/Documents/JSRP/privia/corpus  (3 JSON)
✅ Prompts dir: /Users/julissa.rodriguez/Documents/JSRP/privia/prompts  (4 MD)
   corpus → ['ley_21719.json', 'nist_cswp40.json', 'politica_banco_general.json']
   prompts → ['auditor.md', 'fiscalizador.md', 'orquestador.md', 'query_expansion.md']


## 📦 CELDA 1 — Instalación de dependencias

In [28]:
# ─── Instalación de dependencias (usa el kernel ACTUAL, no el pip del sistema) ─
# %pip install instala siempre al intérprete del kernel.
# !pip install puede instalar al Python equivocado y dar ModuleNotFoundError.
%pip install -q openai redis langgraph langchain-core langchain-openai \
              langgraph-checkpoint-sqlite numpy pydantic python-dotenv

print('✅ Dependencias instaladas en el kernel actual:')
print('   • openai          → LLM GPT-4o + embeddings text-embedding-3-small (1536 dims)')
print('   • redis           → Vector Store normativo con claves normativa:doc/emb/index')
print('   • langgraph       → Orquestación del grafo Multi-Agente')
print('   • numpy           → Similitud coseno KNN')
print('   • python-dotenv   → Carga .env al correr local (no hace nada en Colab)')
print()


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencias instaladas en el kernel actual:
   • openai          → LLM GPT-4o + embeddings text-embedding-3-small (1536 dims)
   • redis           → Vector Store normativo con claves normativa:doc/emb/index
   • langgraph       → Orquestación del grafo Multi-Agente
   • numpy           → Similitud coseno KNN
   • python-dotenv   → Carga .env al correr local (no hace nada en Colab)



## 🔑 CELDA 2 — Configuración de credenciales (Secrets)

Las credenciales **nunca** se escriben en el notebook. El orden de precedencia es:

1. **Google Colab Secrets** — ícono 🔑 en el panel izquierdo (recomendado en Colab).
2. **Variables de entorno** — útil al correr local (`export OPENAI_API_KEY=…` o un `.env`).
3. **`getpass`** — si nada de lo anterior está definido, se pide interactivamente y *no* queda persistido.

| Secret name | Descripción |
|---|---|
| `OPENAI_API_KEY` | Clave de OpenAI (LLM + embeddings) |
| `REDIS_HOST` | Host de Redis Cloud — ej. `redis-12345.redislabs.com` |
| `REDIS_PORT` | Puerto Redis — ej. `14159` |
| `REDIS_PASSWORD` | Contraseña Redis Cloud |

> **Redis Cloud gratuito:** https://redis.io/try-free/ → crea una BD y copia el endpoint.

> ⚠️ **Si alguna vez pegaste tu API key directamente en un notebook anterior, considérala comprometida.** Revócala en https://platform.openai.com/api-keys y vuelve a crearla como Colab Secret.


In [29]:
import os
import redis
from getpass import getpass
from dotenv import load_dotenv

# Si corres local con un .env, cárgalo. En Colab este import simplemente no aplica.
try:
    load_dotenv()
except Exception:
    pass

# ─── Resolución de credenciales: Colab Secrets → env vars → prompt ──────────
def _get_secret(name, *, prompt=None, cast=str, required=True):
    """Lee una credencial de Colab Secrets, luego env vars, luego pide por consola."""
    value = None
    try:
        from google.colab import userdata  # solo existe en Colab
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    except ImportError:
        pass
    if not value:
        value = os.environ.get(name)
    if not value and required:
        value = getpass(prompt or f'{name}: ')
    return cast(value) if value else None


OPENAI_API_KEY = _get_secret('OPENAI_API_KEY', prompt='OpenAI API key (sk-...): ')
REDIS_HOST     = _get_secret('REDIS_HOST',     prompt='Redis host: ')
REDIS_PORT     = _get_secret('REDIS_PORT',     prompt='Redis port: ', cast=int)
REDIS_PASSWORD = _get_secret('REDIS_PASSWORD', prompt='Redis password: ')

# Exportar OPENAI_API_KEY al entorno para que el SDK la levante automáticamente.
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ─── Conexión Redis (cliente global) ────────────────────────────────────────
# decode_responses=False → necesario para manejar bytes de embeddings float32
REDIS_CLIENT = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    decode_responses=False,
    socket_connect_timeout=5,
)

try:
    REDIS_CLIENT.ping()
    info = REDIS_CLIENT.info('server')
    docs_indexados = REDIS_CLIENT.scard('normativa:index')
    print(f'✅ OpenAI API Key  : {OPENAI_API_KEY[:8]}…{OPENAI_API_KEY[-4:]}')
    # Enmascarar el hostname en el output para que no quede en el .ipynb pusheado.
    _host = REDIS_HOST or ''
    _host_masked = (_host[:8] + '…' + _host[-15:]) if len(_host) > 25 else _host
    print(f'✅ Redis conectado : {_host_masked}:{REDIS_PORT}')
    print(f'   Redis version  : {info["redis_version"]}')
    print(f'   Docs indexados : {docs_indexados} (0 = indexación pendiente)')
except Exception as e:
    print(f'❌ Error de conexión Redis: {e}')
    print('   Revisa REDIS_HOST, REDIS_PORT y REDIS_PASSWORD')


✅ OpenAI API Key  : sk-proj-…JrYA
✅ Redis conectado : redis-14…d.redislabs.com:14159
   Redis version  : 8.4.0
   Docs indexados : 23 (0 = indexación pendiente)


## 📋 CELDA 3 — Base de Conocimiento Normativa (RAG con Redis)

Pipeline de indexación según especificación PRIVIA:

```
Documento normativo
      ↓
Chunking por artículo/sección  (source + article + section + content)
      ↓
OpenAI text-embedding-3-small  (vector 1536 dims)
      ↓
Redis Storage
  normativa:doc:<md5>  →  JSON con metadata completa
  normativa:emb:<md5>  →  bytes float32 del vector
  normativa:index      →  SET de doc_ids
```

> La indexación se ejecuta **una sola vez**. Si Redis ya tiene chunks (`normativa:index` no vacío),
> se salta automáticamente para no re-indexar ni duplicar.

In [30]:
import os
import json
import hashlib
import numpy as np
from pathlib import Path
from typing import List, Literal
from datetime import date, datetime
from collections import Counter
from pydantic import BaseModel, Field, field_validator
from openai import OpenAI

# ── Configuración OpenAI / RAG ──────────────────────────────────────────────
OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL   = 'text-embedding-3-small'   # 1536 dimensiones
EMBED_DIMS    = 1536
RAG_TOP_K     = 5
SCORE_UMBRAL  = 0.45   # scores < 0.45 → weak_citation → ISSUES (calibrado al corpus)

# ── Ruta del corpus ─────────────────────────────────────────────────────────
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd()

CORPUS_DIR = Path(os.environ.get('PRIVIA_CORPUS_DIR', BASE_DIR / 'corpus'))


# ── Schema con validación ───────────────────────────────────────────────────
class NormativaChunk(BaseModel):
    document_name: str
    document_type: Literal['ley', 'framework', 'politica_interna']
    article: str
    section: str
    content: str = Field(min_length=50)
    document_uri: str
    chunk_id: str = Field(pattern=r'^[a-z0-9_]+$')
    criticality: Literal['high', 'medium', 'low']
    topic: str
    source_owner: str
    updated_at: date

    @field_validator('content')
    @classmethod
    def _content_not_blank(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('content vacío')
        return v


# ── Loader ──────────────────────────────────────────────────────────────────
def load_corpus(corpus_dir: Path) -> List[NormativaChunk]:
    if not corpus_dir.is_dir():
        raise FileNotFoundError(f'No existe el directorio de corpus: {corpus_dir}')

    chunks: List[NormativaChunk] = []
    seen_ids: set[str] = set()

    for file in sorted(corpus_dir.glob('*.json')):
        with file.open('r', encoding='utf-8') as f:
            payload = json.load(f)

        for raw in payload['chunks']:
            chunk = NormativaChunk(**raw)
            if chunk.chunk_id in seen_ids:
                raise ValueError(f'chunk_id duplicado: {chunk.chunk_id} en {file.name}')
            seen_ids.add(chunk.chunk_id)
            chunks.append(chunk)

    if not chunks:
        raise RuntimeError(f'Corpus vacío en {corpus_dir}')
    return chunks


# ── Carga ───────────────────────────────────────────────────────────────────
CORPUS_NORMATIVO = load_corpus(CORPUS_DIR)

print(f'📚 Corpus normativo cargado: {len(CORPUS_NORMATIVO)} chunks desde {CORPUS_DIR}')
for fuente, n in Counter(c.document_name for c in CORPUS_NORMATIVO).items():
    print(f'   • {fuente}: {n} chunks')

📚 Corpus normativo cargado: 23 chunks desde /Users/julissa.rodriguez/Documents/JSRP/privia/corpus
   • Ley 21.719 — Protección y Tratamiento de Datos Personales: 10 chunks
   • NIST Privacy Framework 1.1 (CSWP 40): 8 chunks
   • Política de Protección de Datos Personales — Banco General: 5 chunks


## 🗄️ CELDA 4 — Indexación en Redis y Motor de Búsqueda KNN

**Pipeline de indexación:**
1. Para cada chunk: genera texto `source + article + section + content`
2. Llama a `text-embedding-3-small` → vector float32 de 1536 dims
3. Guarda en Redis:
   - `normativa:doc:<md5>` → JSON con metadata completa
   - `normativa:emb:<md5>` → bytes float32 del vector
   - `normativa:index` → SADD del doc_id

**Búsqueda KNN:** similitud coseno entre query embedding y todos los embeddings del índice.

In [31]:
from dataclasses import dataclass
import struct
import time

REDIS_INDEX_KEY = 'normativa:index'   # SET con todos los doc_ids

# ════════════════════════════════════════════════════════════════════════════
#  HELPERS: serialización de vectores float32 ↔ bytes
# ════════════════════════════════════════════════════════════════════════════
def vector_to_bytes(vec: List[float]) -> bytes:
    """Serializa lista de floats a bytes little-endian (float32)."""
    return struct.pack(f'{len(vec)}f', *vec)

def bytes_to_vector(b: bytes) -> List[float]:
    """Deserializa bytes a lista de floats."""
    n = len(b) // 4
    return list(struct.unpack(f'{n}f', b))

def cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    similitud(A, B) = (A · B) / (||A|| × ||B||)
    Implementación de la similitud coseno según especificación PRIVIA.
    """
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    norm_a = np.linalg.norm(va)
    norm_b = np.linalg.norm(vb)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(va, vb) / (norm_a * norm_b))

# ════════════════════════════════════════════════════════════════════════════
#  GENERACIÓN DE EMBEDDINGS (OpenAI text-embedding-3-small)
# ════════════════════════════════════════════════════════════════════════════
def embed_text(text: str) -> List[float]:
    """Genera embedding de 1536 dims con text-embedding-3-small."""
    resp = OPENAI_CLIENT.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp.data[0].embedding

# ════════════════════════════════════════════════════════════════════════════
#  INDEXACIÓN EN REDIS
#  Ejecuta una sola vez. Si normativa:index ya tiene docs, omite.
# ════════════════════════════════════════════════════════════════════════════
def indexar_normativa(forzar: bool = False) -> int:
    """
    Indexa el corpus normativo en Redis.
    Claves generadas por chunk:
      normativa:doc:<md5>  → JSON con metadata completa
      normativa:emb:<md5>  → bytes float32 del vector (1536 dims)
      normativa:index      → SET con todos los chunk_ids
    Retorna número de chunks indexados.
    """
    ya_indexado = REDIS_CLIENT.scard(REDIS_INDEX_KEY)
    if ya_indexado > 0 and not forzar:
        print(f'ℹ️  Redis ya tiene {ya_indexado} chunks indexados. Omitiendo indexación.')
        print('   Usa indexar_normativa(forzar=True) para re-indexar.')
        return ya_indexado

    print(f'⏳ Iniciando indexación de {len(CORPUS_NORMATIVO)} chunks en Redis...')
    print(f'   Modelo: {EMBED_MODEL} ({EMBED_DIMS} dims)')
    print(f'   Claves: normativa:doc:<md5> | normativa:emb:<md5> | normativa:index')
    print()

    indexados = 0
    t_inicio = time.time()

    for i, chunk in enumerate(CORPUS_NORMATIVO, 1):
        # 1. Texto de indexación: source + article + section + content
        texto_indexar = (
            f"{chunk.document_name} | {chunk.article} | "
            f"{chunk.section} | {chunk.content}"
        )

        # 2. MD5 como identificador de chunk
        doc_id = hashlib.md5(texto_indexar.encode()).hexdigest()

        # 3. Generar embedding (text-embedding-3-small)
        vector = embed_text(texto_indexar)

        # 4. Metadata completa para normativa:doc:<md5>
        metadata = {
            'doc_id'        : doc_id,
            'chunk_id'      : chunk.chunk_id,
            'document_name' : chunk.document_name,
            'document_type' : chunk.document_type,
            'article'       : chunk.article,
            'section'       : chunk.section,
            'content'       : chunk.content,
            'document_uri'  : chunk.document_uri,
            'criticality'   : chunk.criticality,
            'topic'         : chunk.topic,
            'source_owner'  : chunk.source_owner,
            'updated_at'    : chunk.updated_at.isoformat(), # Convert date to ISO format string
            'embed_model'   : EMBED_MODEL,
            'embed_dims'    : EMBED_DIMS,
            'indexed_at'    : datetime.now().isoformat()
        }

        # 5. Persistir en Redis
        pipe = REDIS_CLIENT.pipeline()
        pipe.set(f'normativa:doc:{doc_id}', json.dumps(metadata, ensure_ascii=False))
        pipe.set(f'normativa:emb:{doc_id}', vector_to_bytes(vector))
        pipe.sadd(REDIS_INDEX_KEY, doc_id)
        pipe.execute()

        indexados += 1
        print(f'   [{i:02d}/{len(CORPUS_NORMATIVO)}] ✓ {chunk.chunk_id} [{chunk.document_type}]')

        # Pausa para respetar rate limit de OpenAI
        if i < len(CORPUS_NORMATIVO):
            time.sleep(0.3)

    t_total = time.time() - t_inicio
    print(f'\n✅ Indexación completada: {indexados} chunks en {t_total:.1f}s')
    print(f'   Redis (normativa:index): {REDIS_CLIENT.scard(REDIS_INDEX_KEY)} doc_ids')
    return indexados


# ════════════════════════════════════════════════════════════════════════════
#  BÚSQUEDA KNN — tool_search_normativa
# ════════════════════════════════════════════════════════════════════════════
@dataclass
class LawReference:
    """Formato de cita estructurada retornado por el Worker RAG."""
    source         : str
    article        : str
    section        : str
    content        : str
    retrieval_score: float
    document_uri   : str
    criticality    : str
    topic          : str


def tool_search_normativa(query: str, top_k: int = RAG_TOP_K) -> List[LawReference]:
    """
    Worker RAG Normativo: búsqueda semántica KNN sobre Redis.

    Algoritmo:
      1. Genera embedding de la query con text-embedding-3-small
      2. Carga todos los doc_ids desde normativa:index
      3. Para cada doc_id: recupera normativa:emb:<md5> y calcula similitud coseno
      4. Retorna top-K como objetos LawReference con retrieval_score

    Nota: si retrieval_score < SCORE_UMBRAL (0.45) → weak_citation → ISSUES
    """
    # 1. Embedding de la consulta
    query_emb = embed_text(query)

    # 2. Cargar todos los doc_ids del índice
    doc_ids = REDIS_CLIENT.smembers(REDIS_INDEX_KEY)
    if not doc_ids:
        print('⚠️  normativa:index vacío — ejecuta indexar_normativa() primero')
        return []

    # 3. Calcular similitud coseno con cada embedding almacenado
    scores = []
    for raw_id in doc_ids:
        doc_id = raw_id.decode() if isinstance(raw_id, bytes) else raw_id
        emb_bytes = REDIS_CLIENT.get(f'normativa:emb:{doc_id}')
        if emb_bytes is None:
            continue
        stored_emb = bytes_to_vector(emb_bytes)
        sim = cosine_similarity(query_emb, stored_emb)
        scores.append((sim, doc_id))

    # 4. Top-K por similitud coseno (mayor es mejor)
    scores.sort(key=lambda x: x[0], reverse=True)
    top_docs = scores[:top_k]

    # 5. Construir LawReferences desde metadata Redis
    resultados = []
    for sim, doc_id in top_docs:
        if sim < 0.15:   # filtro de ruido absoluto
            continue
        meta_raw = REDIS_CLIENT.get(f'normativa:doc:{doc_id}')
        if meta_raw is None:
            continue
        meta = json.loads(meta_raw.decode())
        resultados.append(LawReference(
            source          = meta['document_name'],
            article         = meta['article'],
            section         = meta['section'],
            content         = meta['content'],
            retrieval_score = round(sim, 4),
            document_uri    = meta['document_uri'],
            criticality     = meta['criticality'],
            topic           = meta['topic']
        ))

    return resultados


def ver_estado_redis() -> None:
    """Muestra el estado actual del índice normativo en Redis."""
    total = REDIS_CLIENT.scard(REDIS_INDEX_KEY)
    print(f'📊 Estado Redis — índice normativa_interna')
    print(f'   Clave índice  : {REDIS_INDEX_KEY}')
    print(f'   Total chunks  : {total}')
    if total > 0:
        muestra_id = list(REDIS_CLIENT.smembers(REDIS_INDEX_KEY))[0]
        muestra_id = muestra_id.decode() if isinstance(muestra_id, bytes) else muestra_id
        meta_raw = REDIS_CLIENT.get(f'normativa:doc:{muestra_id}')
        if meta_raw:
            meta = json.loads(meta_raw.decode())
            print(f'   Ejemplo doc   : {meta["chunk_id"]} [{meta["document_type"]}]')
            print(f'   Embed model   : {meta["embed_model"]} ({meta["embed_dims"]} dims)')
            print(f'   Indexed at    : {meta["indexed_at"]}')


# ─── EJECUTAR INDEXACIÓN ────────────────────────────────────────────────────
# ⚠️  Primera ejecución: llama a OpenAI para cada chunk (~20 llamadas × 0.3s)
# Tiempo estimado: 30-60 segundos. Se salta automáticamente si ya está indexado.
indexar_normativa()
print()
ver_estado_redis()


ℹ️  Redis ya tiene 23 chunks indexados. Omitiendo indexación.
   Usa indexar_normativa(forzar=True) para re-indexar.

📊 Estado Redis — índice normativa_interna
   Clave índice  : normativa:index
   Total chunks  : 23
   Ejemplo doc   : nist_ct_dm_p [framework]
   Embed model   : text-embedding-3-small (1536 dims)
   Indexed at    : 2026-06-01T21:33:55.398259


## 🧪 CELDA 5 — Test del Motor RAG

In [32]:
print('🧪 Test del motor RAG — Búsqueda KNN en Redis')
print('   Similitud coseno: similitud(A,B) = (A·B) / (||A|| × ||B||)')
print('   Umbral weak_citation: retrieval_score < 0.60 → ISSUES\n')

QUERY_TEST = 'almacenamiento de datos biométricos para autenticación de clientes'
print(f'Query: "{QUERY_TEST}"\n')

resultados_test = tool_search_normativa(QUERY_TEST, top_k=5)
for i, r in enumerate(resultados_test, 1):
    estado = '✓ relevante' if r.retrieval_score >= SCORE_UMBRAL else '⚠ weak_citation'
    print(f'  [{i}] score={r.retrieval_score:.2f} {estado}')
    print(f'      Fuente : {r.source}')
    print(f'      Art.   : {r.article}')
    print(f'      Sección: {r.section}')
    print(f'      URI    : {r.document_uri}')
    print()

print(f'✅ RAG operativo: {len(resultados_test)} hits encontrados')

🧪 Test del motor RAG — Búsqueda KNN en Redis
   Similitud coseno: similitud(A,B) = (A·B) / (||A|| × ||B||)
   Umbral weak_citation: retrieval_score < 0.60 → ISSUES

Query: "almacenamiento de datos biométricos para autenticación de clientes"

  [1] score=0.53 ✓ relevante
      Fuente : Política de Protección de Datos Personales — Banco General
      Art.   : Sección 6
      Sección: Medidas de seguridad técnicas y organizativas
      URI    : data/politica_proteccion_datos.pdf#sec6

  [2] score=0.52 ✓ relevante
      Fuente : Política de Protección de Datos Personales — Banco General
      Art.   : Sección 5
      Sección: Retención, conservación y eliminación de datos
      URI    : data/politica_proteccion_datos.pdf#sec5

  [3] score=0.50 ✓ relevante
      Fuente : Ley 21.719 — Protección y Tratamiento de Datos Personales
      Art.   : Art. 16
      Sección: Tratamiento de datos personales sensibles
      URI    : data/ley_21719.pdf#art16

  [4] score=0.42 ⚠ weak_citation
      Fuent

## 🗄️ CELDA 6 — Worker Data Catalog (simulado) + Input Sanitizer

In [33]:
import re
from typing import Dict

# ════════════════════════════════════════════════════════════════════════════
#  WORKER DATA CATALOG
#  Identifica sensibilidad, PII, dueños de datos y ubicación
# ════════════════════════════════════════════════════════════════════════════
@dataclass
class DataCatalogEntry:
    table        : str
    field        : str
    pii          : bool
    sensitivity  : str   # low | medium | high | critical
    data_type    : str   # personal | biometric | financial | health | behavioral
    owner        : str
    location     : str
    retention_days: int

DATA_CATALOG: List[DataCatalogEntry] = [
    DataCatalogEntry('usuarios',     'rut',               True,  'critical', 'personal',   'equipo_identidad', 'cloud_aws_us_east_1', 1825),
    DataCatalogEntry('usuarios',     'nombre_completo',   True,  'high',     'personal',   'equipo_identidad', 'cloud_aws_us_east_1', 1825),
    DataCatalogEntry('usuarios',     'email',             True,  'high',     'personal',   'equipo_crm',       'cloud_aws_us_east_1', 1825),
    DataCatalogEntry('autenticacion','huella_hash',       True,  'critical', 'biometric',  'equipo_seguridad', 'on_premise_sgd',      365),
    DataCatalogEntry('autenticacion','face_vector',       True,  'critical', 'biometric',  'equipo_seguridad', 'on_premise_sgd',      365),
    DataCatalogEntry('transacciones','monto',             True,  'high',     'financial',  'equipo_finanzas',  'cloud_azure_eastus',  2555),
    DataCatalogEntry('transacciones','cuenta_origen',     True,  'critical', 'financial',  'equipo_finanzas',  'cloud_azure_eastus',  2555),
    DataCatalogEntry('logs_acceso',  'ip_address',        True,  'medium',   'behavioral', 'equipo_soc',       'cloud_aws_us_east_1', 90),
    DataCatalogEntry('perfiles_ia',  'score_credito',     True,  'critical', 'financial',  'equipo_ia',        'cloud_aws_us_east_1', 180),
    DataCatalogEntry('perfiles_ia',  'vector_comportamiento', True, 'high', 'behavioral', 'equipo_ia',        'cloud_aws_us_east_1', 180),
    DataCatalogEntry('salud',        'diagnostico',       True,  'critical', 'health',     'equipo_salud',     'on_premise_sgd',      3650),
    DataCatalogEntry('cookies',      'session_id',        False, 'low',      'behavioral', 'equipo_web',       'cdn_cloudflare',      30),
]

def tool_query_catalog(keywords: List[str]) -> Dict:
    """Worker SQL/Data Catalog: identifica sensibilidad, PII, dueños y ubicación."""
    kw = [k.lower() for k in keywords]
    matches = [
        e for e in DATA_CATALOG
        if any(w in e.table.lower() or w in e.field.lower() or
               w in e.data_type.lower() or w in e.sensitivity.lower()
               for w in kw)
    ]
    pii_fields      = [e for e in matches if e.pii]
    critical_fields = [e for e in matches if e.sensitivity == 'critical']
    cloud_locations = {e.location for e in matches if 'cloud' in e.location}
    return {
        'total_fields_found'  : len(matches),
        'pii_fields_count'    : len(pii_fields),
        'critical_fields_count': len(critical_fields),
        'sensitive_data_types': list({e.data_type for e in matches if e.pii}),
        'cloud_locations'     : list(cloud_locations),
        'has_biometric_data'  : any(e.data_type == 'biometric' for e in matches),
        'has_financial_data'  : any(e.data_type == 'financial' for e in matches),
        'has_health_data'     : any(e.data_type == 'health' for e in matches),
        'has_ai_profiling'    : any('ia' in e.table or 'perfil' in e.table for e in matches),
        'owners'              : list({e.owner for e in matches}),
        'retention_max_days'  : max((e.retention_days for e in matches), default=0),
        'sample_entries'      : [
            {'table': e.table, 'field': e.field, 'pii': e.pii,
             'sensitivity': e.sensitivity, 'type': e.data_type, 'location': e.location}
            for e in matches[:6]
        ]
    }

# ════════════════════════════════════════════════════════════════════════════
#  INPUT SANITIZER — Paso 0 del pipeline
#  Detecta PII real y aplica scrubbing antes de enviar al pipeline
# ════════════════════════════════════════════════════════════════════════════
PII_PATTERNS = {
    'rut_chileno'    : r'\b\d{7,8}-[\dkK]\b',
    'email'          : r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
    'telefono_cl'    : r'\b(\+56|56)?\s?[9]\d{8}\b',
    'tarjeta_credito': r'\b(?:\d{4}[\s-]?){3}\d{4}\b',
    'ip_address'     : r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
    'passport'       : r'\b[A-Z]{1,2}\d{6,9}\b',
}

def input_sanitizer(texto: str) -> Dict:
    """
    Paso 0: detecta y elimina PII real del input.
    Si detecta PII: aplica scrubbing y setea pii_scrubbed=True.
    """
    pii_found = {}
    texto_limpio = texto
    for nombre, patron in PII_PATTERNS.items():
        if re.search(patron, texto):
            pii_found[nombre] = re.findall(patron, texto)
            texto_limpio = re.sub(patron, f'[{nombre.upper()}_REDACTED]', texto_limpio)
    return {
        'status'      : 'scrubbed' if pii_found else 'clean',
        'pii_scrubbed': bool(pii_found),
        'texto_limpio': texto_limpio,
        'pii_detected': list(pii_found.keys()),
        'advertencia' : f'⚠️ PII detectada y eliminada: {list(pii_found.keys())}' if pii_found else None
    }

print('✅ Worker Data Catalog listo:', len(DATA_CATALOG), 'entradas')
print('✅ Input Sanitizer listo')

# Test rápido
test_san = input_sanitizer('Sistema para RUT 12345678-9 con email admin@banco.com')
print(f'\n🧪 Test Sanitizer: PII detectada={test_san["pii_detected"]} → "{test_san["texto_limpio"][:60]}..."')

✅ Worker Data Catalog listo: 12 entradas
✅ Input Sanitizer listo

🧪 Test Sanitizer: PII detectada=['rut_chileno', 'email'] → "Sistema para RUT [RUT_CHILENO_REDACTED] con email [EMAIL_RED..."


## 🤖 CELDA 7 — Agentes LLM con LangGraph (Orquestador, Auditor, Fiscalizador)

In [34]:
from pathlib import Path
import operator
import uuid
import re
from dataclasses import asdict
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, AnyMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver # Import InMemorySaver

# ════════════════════════════════════════════════════════════════════════════
#  ESTADO DEL PIPELINE DE AUDITORÍA
# ════════════════════════════════════════════════════════════════════════════
class AuditState(TypedDict):
    messages           : Annotated[list[AnyMessage], operator.add]
    consulta_original  : str
    pii_scrubbed       : bool
    consulta_limpia    : str
    # Clasificación del orquestador
    query_type         : str      # legal | technical | complex | incomplete | validation_only
    tools_to_invoke    : list
    orquestador_razon  : str
    # Evidencia recopilada
    rag_hits           : list     # List[dict] con LawReferences
    catalog_result     : dict
    evidence_gap       : bool
    weak_citations     : list     # hits con retrieval_score < SCORE_UMBRAL
    # Reportes
    reporte_preliminar : str
    reporte_final      : str
    audit_status       : str      # OK | ISSUES | CORRECTED | INCOMPLETE
    findings           : list
    # Trazabilidad
    audit_id           : str
    timestamp          : str

# ── Modelos LLM ──────────────────────────────────────────────────────────────
LLM_ORQUESTADOR  = ChatOpenAI(model='gpt-4o', temperature=0.1)
LLM_AUDITOR      = ChatOpenAI(model='gpt-4o', temperature=0.2)
LLM_FISCALIZADOR = ChatOpenAI(model='gpt-4o', temperature=0.0)
LLM_QUERY_EXPANDER = ChatOpenAI(model='gpt-4o-mini', temperature=0.0)  # FIX 2: query expansion

# ════════════════════════════════════════════════════════════════════════════
#  Prompts externalizados a prompts/*.md (cargados al inicio del módulo)
#  Mantenerlos como archivos permite versionarlos como contenido y editarlos
#  sin tocar el notebook. Si tienen placeholders {SCORE_UMBRAL}, etc., se
#  resuelven con .format() al construirlos.
# ════════════════════════════════════════════════════════════════════════════
PROMPTS_DIR = Path(os.environ.get('PRIVIA_PROMPTS_DIR', Path.cwd() / 'prompts'))

def _load_prompt(filename: str, **fmt) -> str:
    path = PROMPTS_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'No encontré el prompt: {path}')
    raw = path.read_text(encoding='utf-8')
    return raw.format(**fmt) if fmt else raw

PROMPT_QUERY_EXPANSION = _load_prompt('query_expansion.md')
PROMPT_ORQUESTADOR     = _load_prompt('orquestador.md')
PROMPT_AUDITOR         = _load_prompt('auditor.md')
PROMPT_FISCALIZADOR    = _load_prompt('fiscalizador.md', SCORE_UMBRAL=SCORE_UMBRAL)

# ════════════════════════════════════════════════════════════════════════════
#  HELPER: QUERY EXPANSION (FIX 2)
#  Reescribe una descripción de arquitectura como términos jurídico-técnicos
#  para mejorar el match contra el corpus normativo (sube scores ~0.15-0.25).
# ════════════════════════════════════════════════════════════════════════════

def expand_query(consulta: str) -> str:
    """Reescribe una descripción técnica como términos jurídico-técnicos."""
    resp = LLM_QUERY_EXPANDER.invoke([
        SystemMessage(content=PROMPT_QUERY_EXPANSION),
        HumanMessage(content=f'Arquitectura/consulta:\n{consulta}')
    ])
    return resp.content.strip()

# ════════════════════════════════════════════════════════════════════════════
#  HELPER: DETECCIÓN PROGRAMÁTICA DE PII REAL EN UN TEXTO (FIX 3)
#  Usa los mismos PII_PATTERNS del sanitizador. Distingue VALOR (leak)
#  de CATEGORÍA (mencionar la palabra "RUT" o "email" no es leak).
# ════════════════════════════════════════════════════════════════════════════
def detectar_pii_valores(texto: str) -> dict:
    """Retorna {tipo_pii: [valores]} para cada VALOR real detectado."""
    encontrados = {}
    for nombre, patron in PII_PATTERNS.items():
        matches = re.findall(patron, texto)
        # Filtrar los placeholders del sanitizador: [RUT_CHILENO_REDACTED], etc.
        matches = [m for m in matches if 'REDACTED' not in m]
        if matches:
            encontrados[nombre] = matches
    return encontrados

# ════════════════════════════════════════════════════════════════════════════
#  NODO 0: INPUT SANITIZER
# ════════════════════════════════════════════════════════════════════════════
def nodo_sanitizador(state: AuditState) -> dict:
    result = input_sanitizer(state['consulta_original'])
    msgs = []
    if result['advertencia']:
        msgs.append(AIMessage(content=result['advertencia']))
    return {
        'messages'      : msgs,
        'pii_scrubbed'  : result['pii_scrubbed'],
        'consulta_limpia': result['texto_limpio'],
        'audit_id'      : str(uuid.uuid4())[:8].upper(),
        'timestamp'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

# ════════════════════════════════════════════════════════════════════════════
#  NODO 2: ORQUESTADOR — Clasificador LLM + decisión de tools
#  Criterios: legal→rag | technical→catalog | complex→ambas | incomplete→abort
# ════════════════════════════════════════════════════════════════════════════

def nodo_orquestador(state: AuditState) -> dict:
    resp = LLM_ORQUESTADOR.invoke([
        SystemMessage(content=PROMPT_ORQUESTADOR),
        HumanMessage(content=f'Consulta a clasificar:\n{state["consulta_limpia"]}')
    ])
    try:
        content = resp.content.strip().replace('```json','').replace('```','').strip()
        clf = json.loads(content)
    except Exception:
        clf = {'type': 'complex', 'tools_to_invoke': ['rag','catalog'], 'reason': 'Fallback: error de parseo'}

    msg = AIMessage(content=(
        f'[ORQUESTADOR] tipo={clf.get("type")} | '
        f'tools={clf.get("tools_to_invoke")} | '
        f'razón: {clf.get("reason")}'
    ))
    return {
        'messages'          : [msg],
        'query_type'        : clf.get('type', 'complex'),
        'tools_to_invoke'   : clf.get('tools_to_invoke', ['rag','catalog']),
        'orquestador_razon' : clf.get('reason', '')
    }

def routing_orquestador(state: AuditState) -> str:
    return 'nodo_incompleto' if state.get('query_type') == 'incomplete' else 'nodo_workers'

def nodo_incompleto(state: AuditState) -> dict:
    msg = AIMessage(content=(
        '[PRIVIA] Consulta incompleta. Se requieren antecedentes técnicos: '
        'arquitectura de la solución, tipos de datos involucrados, tecnologías '
        '(cloud, APIs, IA), y contexto regulatorio aplicable.'
    ))
    return {'messages': [msg], 'audit_status': 'INCOMPLETE',
            'reporte_final': 'Consulta incompleta. Se requieren más antecedentes.'}

# ════════════════════════════════════════════════════════════════════════════
#  NODO 3: WORKERS — RAG Redis + Data Catalog
#  Si rag_hits=0 → evidence_gap=True → fuerza audit_status=ISSUES
# ════════════════════════════════════════════════════════════════════════════
def nodo_workers(state: AuditState) -> dict:
    tools   = state.get('tools_to_invoke', ['rag','catalog'])
    consulta = state['consulta_limpia']
    msgs, rag_hits, catalog_result, weak_citations = [], [], {}, []

    # Worker RAG — búsqueda KNN en Redis con query expansion (FIX 2)
    if 'rag' in tools:
        query_expandida = expand_query(consulta)
        msgs.append(AIMessage(content=(
            f'[QUERY_EXPANSION] {query_expandida[:140]}'
            f"{'...' if len(query_expandida) > 140 else ''}"
        )))
        hits = tool_search_normativa(query_expandida, top_k=RAG_TOP_K)
        rag_hits = [asdict(h) for h in hits]
        weak_citations = [h for h in rag_hits if h['retrieval_score'] < SCORE_UMBRAL]
        top_score = max((h['retrieval_score'] for h in rag_hits), default=0.0)
        msgs.append(AIMessage(content=(
            f'[WORKER_RAG] {len(hits)} hits | top_score={top_score:.3f} | '
            f'weak_citations: {len(weak_citations)} '
            f'(score < {SCORE_UMBRAL})'
        )))

    # Worker Data Catalog
    if 'catalog' in tools:
        kw = consulta.lower().split()[:15]
        catalog_result = tool_query_catalog(kw)
        msgs.append(AIMessage(content=(
            f'[WORKER_CATALOG] campos={catalog_result["total_fields_found"]} | '
            f'PII={catalog_result["pii_fields_count"]} | '
            f'críticos={catalog_result["critical_fields_count"]}'
        )))

    evidence_gap = len(rag_hits) == 0
    if evidence_gap:
        msgs.append(AIMessage(content='[WORKER_RAG] ⚠️ evidence_gap=True → audit_status forzado a ISSUES'))

    return {
        'messages'      : msgs,
        'rag_hits'      : rag_hits,
        'catalog_result': catalog_result,
        'evidence_gap'  : evidence_gap,
        'weak_citations': weak_citations
    }

# ════════════════════════════════════════════════════════════════════════════
#  NODO 4: LLM AUDITOR PRINCIPAL (GPT-4o)
#  Integra evidencia RAG + Catalog y genera reporte preliminar
# ════════════════════════════════════════════════════════════════════════════

def nodo_auditor(state: AuditState) -> dict:
    rag_texto = '\n'.join([
        f"- [{r['source']}] {r['article']} | {r['section']} "
        f"(score={r['retrieval_score']}) | URI: {r['document_uri']}\n  {r['content'][:300]}..."
        for r in state.get('rag_hits', [])
    ]) or 'SIN EVIDENCIA NORMATIVA (evidence_gap=True)'

    catalog_texto = json.dumps(state.get('catalog_result', {}), ensure_ascii=False, indent=2)

    prompt_usuario = f"""CONSULTA:\n{state['consulta_limpia']}

EVIDENCIA RAG (Redis normativa_interna):
{rag_texto}

EVIDENCIA DATA CATALOG:
{catalog_texto}

evidence_gap: {state.get('evidence_gap', False)}
pii_scrubbed: {state.get('pii_scrubbed', False)}
weak_citations: {len(state.get('weak_citations', []))} hits con score < {SCORE_UMBRAL}"""

    resp = LLM_AUDITOR.invoke([
        SystemMessage(content=PROMPT_AUDITOR),
        HumanMessage(content=prompt_usuario)
    ])
    return {
        'messages'          : [AIMessage(content='[AUDITOR_PRINCIPAL] Reporte preliminar generado.')],
        'reporte_preliminar': resp.content
    }

# ════════════════════════════════════════════════════════════════════════════
#  NODO 5: AGENTE FISCALIZADOR (capa independiente de control de calidad)
#  Valida: evidencia, consistencia técnica, PII residual, remediación, formato
#  Veredictos: OK | ISSUES | CORRECTED
# ════════════════════════════════════════════════════════════════════════════

def nodo_fiscalizador(state: AuditState) -> dict:
    reporte_prelim = state.get('reporte_preliminar', '')

    # FIX 3: validación programática de PII en el reporte (valores, no categorías).
    # El sanitizador limpió la consulta, pero el auditor pudo haber introducido PII
    # a partir del Data Catalog. Aquí buscamos VALORES reales (12345678-9, admin@...).
    pii_real_en_reporte = detectar_pii_valores(reporte_prelim)
    pii_real_detectada  = bool(pii_real_en_reporte)

    prompt_usuario = f"""Reporte preliminar a validar:
{reporte_prelim}

Contexto del pipeline:
- evidence_gap: {state.get('evidence_gap', False)}
- rag_hits_count: {len(state.get('rag_hits', []))}
- weak_citations: {len(state.get('weak_citations', []))} (score < {SCORE_UMBRAL})
- pii_scrubbed: {state.get('pii_scrubbed', False)}

VALIDACIÓN PROGRAMÁTICA YA REALIZADA (úsala, no la re-evalúes):
- PII real (valores) detectada en el reporte: {list(pii_real_en_reporte.keys()) if pii_real_detectada else 'ninguna'}

REGLA CRÍTICA PARA EL CONTROL DE PII:
- Mencionar la CATEGORÍA "RUT", "email", "IP" o "nombre" en el reporte NO es pii_leak.
  Es legítimo describir QUÉ TIPOS de datos procesa el sistema.
- Solo es pii_leak si aparece un VALOR específico (ej: 12345678-9, admin@banco.com,
  192.168.1.100, tarjeta 4111 1111 1111 1111).
- La detección de valores reales ya se hizo programáticamente arriba: confía en ese resultado."""

    resp = LLM_FISCALIZADOR.invoke([
        SystemMessage(content=PROMPT_FISCALIZADOR),
        HumanMessage(content=prompt_usuario)
    ])

    reporte_final = resp.content
    audit_status = 'ISSUES'  # default conservador
    if 'VEREDICTO FISCALIZADOR: OK' in reporte_final:
        audit_status = 'OK'
    elif 'VEREDICTO FISCALIZADOR: CORRECTED' in reporte_final:
        audit_status = 'CORRECTED'

    # FIX 4: el fiscalizador NO puede subir el veredicto del auditor.
    # Si el reporte preliminar dijo ISSUES, el final no puede ser OK
    # (a menos que el fiscalizador realmente haya corregido → CORRECTED).
    m_prelim = re.search(r'VEREDICTO PRELIMINAR:\s*(OK|ISSUES)', reporte_prelim, re.IGNORECASE)
    veredicto_auditor = m_prelim.group(1).upper() if m_prelim else None
    if veredicto_auditor == 'ISSUES' and audit_status == 'OK':
        audit_status = 'ISSUES'  # el auditor encontró riesgos sustantivos → respetar

    # evidence_gap, weak_citations o PII real fuerzan ISSUES
    if state.get('evidence_gap') or state.get('weak_citations') or pii_real_detectada:
        audit_status = 'ISSUES'

    fiscal_msgs = [AIMessage(content=f'[FISCALIZADOR] Veredicto final: {audit_status}')]
    if pii_real_detectada:
        fiscal_msgs.append(AIMessage(
            content=f'[FISCALIZADOR] ⚠️ PII real (valores) detectada en el reporte: '
                    f'{list(pii_real_en_reporte.keys())}'
        ))

    return {
        'messages'     : fiscal_msgs,
        'reporte_final': reporte_final,
        'audit_status' : audit_status
    }

# ════════════════════════════════════════════════════════════════════════════
#  CONSTRUCCIÓN DEL GRAFO LANGGRAPH
# ════════════════════════════════════════════════════════════════════════════
memoria = InMemorySaver() # Use InMemorySaver for in-memory checkpointing
grafo   = StateGraph(AuditState)

grafo.add_node('sanitizador',  nodo_sanitizador)
grafo.add_node('orquestador',  nodo_orquestador)
grafo.add_node('incompleto',   nodo_incompleto)
grafo.add_node('workers',      nodo_workers)
grafo.add_node('auditor',      nodo_auditor)
grafo.add_node('fiscalizador', nodo_fiscalizador)

grafo.add_edge(START, 'sanitizador')
grafo.add_edge('sanitizador', 'orquestador')
grafo.add_conditional_edges('orquestador', routing_orquestador,
                            {'nodo_workers': 'workers', 'nodo_incompleto': 'incompleto'})
grafo.add_edge('workers',      'auditor')
grafo.add_edge('auditor',      'fiscalizador')
grafo.add_edge('fiscalizador', END)
grafo.add_edge('incompleto',   END)

PRIVIA = grafo.compile(checkpointer=memoria)

print('✅ Grafo PRIVIA compilado')
print('   START → Sanitizador → Orquestador → Workers (Redis RAG + Catalog) → Auditor → Fiscalizador → END')


✅ Grafo PRIVIA compilado
   START → Sanitizador → Orquestador → Workers (Redis RAG + Catalog) → Auditor → Fiscalizador → END


## 🚀 CELDA 8 — Función de Auditoría + Logger

In [35]:
AUDIT_LOG: List[Dict] = []

def auditar_arquitectura(consulta: str, thread_id: str = None) -> Dict:
    """
    Función principal de PRIVIA.
    Ejecuta el pipeline completo de auditoría de privacidad.
    Retorna: {audit_id, audit_status, reporte_final, log}
    """
    if thread_id is None:
        thread_id = str(uuid.uuid4())[:8]

    config = {'configurable': {'thread_id': thread_id}}

    estado_inicial = {
        'messages'          : [HumanMessage(content=consulta)],
        'consulta_original' : consulta,
        'pii_scrubbed'      : False,
        'consulta_limpia'   : consulta,
        'query_type'        : '',
        'tools_to_invoke'   : [],
        'orquestador_razon' : '',
        'rag_hits'          : [],
        'catalog_result'    : {},
        'evidence_gap'      : False,
        'weak_citations'    : [],
        'reporte_preliminar': '',
        'reporte_final'     : '',
        'audit_status'      : '',
        'findings'          : [],
        'audit_id'          : '',
        'timestamp'         : ''
    }

    print(f'\n{"═"*65}')
    print(f'🔍 PRIVIA — Auditoría de Privacidad   [thread: {thread_id}]')
    print(f'{"═"*65}')

    for evento in PRIVIA.stream(estado_inicial, config=config):
        for nodo_nombre, nodo_estado in evento.items():
            if nodo_nombre == '__end__':
                continue
            if nodo_estado.get('messages'):
                for msg in nodo_estado['messages']:
                    if hasattr(msg, 'content') and msg.content:
                        print(f'  {msg.content[:120]}')

    estado = PRIVIA.get_state(config).values

    # ── Paso 6: Logger de auditoría ──────────────────────────────────────────
    log_entry = {
        'audit_id'          : estado.get('audit_id', thread_id),
        'timestamp'         : estado.get('timestamp', ''),
        'thread_id'         : thread_id,
        'query_type'        : estado.get('query_type', ''),
        'tools_invoked'     : estado.get('tools_to_invoke', []),
        'rag_hits_count'    : len(estado.get('rag_hits', [])),
        'weak_citations'    : len(estado.get('weak_citations', [])),
        'pii_scrubbed'      : estado.get('pii_scrubbed', False),
        'evidence_gap'      : estado.get('evidence_gap', False),
        'audit_status'      : estado.get('audit_status', 'UNKNOWN'),
    }
    AUDIT_LOG.append(log_entry)

    print(f'\n{"─"*65}')
    print(f'✅ Auditoría completada')
    print(f'   ID: {log_entry["audit_id"]}  |  Status: {log_entry["audit_status"]}')
    print(f'   Tipo: {log_entry["query_type"]}  |  RAG hits: {log_entry["rag_hits_count"]}  |  Weak: {log_entry["weak_citations"]}')
    print(f'   PII scrubbed: {log_entry["pii_scrubbed"]}  |  Evidence gap: {log_entry["evidence_gap"]}')
    print(f'{"═"*65}\n')

    return {
        'audit_id'     : log_entry['audit_id'],
        'audit_status' : log_entry['audit_status'],
        'reporte_final': estado.get('reporte_final', ''),
        'log'          : log_entry
    }

print('✅ Función auditar_arquitectura() lista')


✅ Función auditar_arquitectura() lista


## 🧪 CELDA 9 — Casos de Prueba

### CASO 1: Arquitectura de autenticación biométrica en cloud (caso complejo)

In [36]:
CASO_1 = """
Arquitectura de autenticación biométrica para clientes bancarios:
- Se captura huella dactilar y reconocimiento facial en la app móvil
- Los vectores biométricos se almacenan en AWS us-east-1 (fuera de Chile)
- Un modelo de ML genera perfiles de comportamiento para detección de fraude
- Los datos de transacciones se comparten con un proveedor externo de scoring de crédito
- No existe política de retención definida para los vectores biométricos
- No se ha realizado evaluación de impacto de privacidad (DPIA/EIPD)
- Los clientes no han sido informados del procesamiento biométrico
"""

resultado_1 = auditar_arquitectura(CASO_1, thread_id='caso-001')
print(resultado_1['reporte_final'])


═════════════════════════════════════════════════════════════════
🔍 PRIVIA — Auditoría de Privacidad   [thread: caso-001]
═════════════════════════════════════════════════════════════════
  [ORQUESTADOR] tipo=complex | tools=['rag', 'catalog'] | razón: La consulta involucra aspectos legales y técnicos, incluy
  [QUERY_EXPANSION] datos biométricos; tratamiento de datos sensibles; transferencia internacional de datos; falta de base
  [WORKER_RAG] 5 hits | top_score=0.720 | weak_citations: 0 (score < 0.45)
  [WORKER_CATALOG] campos=2 | PII=1 | críticos=1
  [AUDITOR_PRINCIPAL] Reporte preliminar generado.
  [FISCALIZADOR] Veredicto final: ISSUES

─────────────────────────────────────────────────────────────────
✅ Auditoría completada
   ID: 378DB771  |  Status: ISSUES
   Tipo: complex  |  RAG hits: 5  |  Weak: 0
   PII scrubbed: False  |  Evidence gap: False
═════════════════════════════════════════════════════════════════

## REPORTE PRELIMINAR DE PRIVACIDAD — PRIVIA

### 1. RESUMEN EJEC

### CASO 2: Sistema con PII real en el input (scrubbing activo)

In [37]:
CASO_2 = """
Sistema de monitoreo de accesos para usuario RUT 12345678-9, email: admin@banco.com:
- Se registran logs de acceso con IP del usuario: 192.168.1.100
- Los logs incluyen datos de navegación y comportamiento de usuarios
- Retención de logs: 90 días en cloud con cifrado AES-256
- Acceso restringido por roles (RBAC implementado)
- Los logs NO se comparten con terceros
- Existe política de retención documentada y aprobada por el DPO
"""

resultado_2 = auditar_arquitectura(CASO_2, thread_id='caso-002')
print(resultado_2['reporte_final'])


═════════════════════════════════════════════════════════════════
🔍 PRIVIA — Auditoría de Privacidad   [thread: caso-002]
═════════════════════════════════════════════════════════════════
  ⚠️ PII detectada y eliminada: ['rut_chileno', 'email', 'ip_address']
  [ORQUESTADOR] tipo=complex | tools=['rag', 'catalog'] | razón: La consulta involucra aspectos técnicos como logs, IPs, c
  [QUERY_EXPANSION] datos personales; datos de navegación; tratamiento de logs; retención de datos; cifrado de datos; acce
  [WORKER_RAG] 5 hits | top_score=0.643 | weak_citations: 0 (score < 0.45)
  [WORKER_CATALOG] campos=5 | PII=4 | críticos=1
  [AUDITOR_PRINCIPAL] Reporte preliminar generado.
  [FISCALIZADOR] Veredicto final: OK

─────────────────────────────────────────────────────────────────
✅ Auditoría completada
   ID: 2B09C2B4  |  Status: OK
   Tipo: complex  |  RAG hits: 5  |  Weak: 0
   PII scrubbed: True  |  Evidence gap: False
═════════════════════════════════════════════════════════════════

## 

### CASO 3: Consulta incompleta (sin antecedentes técnicos)

In [38]:
CASO_3 = '¿Es nuestro sistema legal?'
resultado_3 = auditar_arquitectura(CASO_3, thread_id='caso-003')
print(resultado_3['reporte_final'])


═════════════════════════════════════════════════════════════════
🔍 PRIVIA — Auditoría de Privacidad   [thread: caso-003]
═════════════════════════════════════════════════════════════════
  [ORQUESTADOR] tipo=incomplete | tools=[] | razón: La consulta es vaga y no proporciona suficiente contexto técnico o leg
  [PRIVIA] Consulta incompleta. Se requieren antecedentes técnicos: arquitectura de la solución, tipos de datos involucrad

─────────────────────────────────────────────────────────────────
✅ Auditoría completada
   ID: C6ECB903  |  Status: INCOMPLETE
   Tipo: incomplete  |  RAG hits: 0  |  Weak: 0
   PII scrubbed: False  |  Evidence gap: False
═════════════════════════════════════════════════════════════════

Consulta incompleta. Se requieren más antecedentes.


## 🎯 CELDA 10 — Consulta personalizada + Resumen del Audit Log

In [39]:
# ─── CONSULTA LIBRE ─────────────────────────────────────────────────────────
# Modifica este texto con tu arquitectura o consulta de privacidad
MI_CONSULTA = """
Plataforma de e-commerce con:
- Datos de pago tokenizados almacenados en Azure
- Sistema de recomendaciones basado en IA que analiza historial de compras
- Cookies de comportamiento compartidas con plataformas publicitarias
- Geolocalización para delivery sin informar al usuario la duración del tratamiento
- No existe delegado de protección de datos designado
"""

# Descomenta para ejecutar:
# resultado_libre = auditar_arquitectura(MI_CONSULTA, thread_id='mi-caso')
# print(resultado_libre['reporte_final'])

# ─── AUDIT LOG ──────────────────────────────────────────────────────────────
print(f'\n📋 AUDIT LOG — Resumen de sesión ({len(AUDIT_LOG)} auditorías)')
print(f'{"─"*80}')
print(f'{"ID":10} {"Timestamp":20} {"Tipo":12} {"Status":12} {"RAG":5} {"Weak":5} {"PII":5}')
print(f'{"─"*80}')
for log in AUDIT_LOG:
    print(
        f'{log["audit_id"]:10} {log["timestamp"]:20} '
        f'{log["query_type"]:12} {log["audit_status"]:12} '
        f'{log["rag_hits_count"]:5} {log["weak_citations"]:5} '
        f'{str(log["pii_scrubbed"]):5}'
    )
print(f'{"─"*80}')

# ─── Estado Redis al cierre ──────────────────────────────────────────────────
print()
ver_estado_redis()


📋 AUDIT LOG — Resumen de sesión (3 auditorías)
────────────────────────────────────────────────────────────────────────────────
ID         Timestamp            Tipo         Status       RAG   Weak  PII  
────────────────────────────────────────────────────────────────────────────────
378DB771   2026-06-01 19:25:15  complex      ISSUES           5     0 False
2B09C2B4   2026-06-01 19:25:38  complex      OK               5     0 True 
C6ECB903   2026-06-01 19:25:56  incomplete   INCOMPLETE       0     0 False
────────────────────────────────────────────────────────────────────────────────

📊 Estado Redis — índice normativa_interna
   Clave índice  : normativa:index
   Total chunks  : 23
   Ejemplo doc   : nist_ct_dm_p [framework]
   Embed model   : text-embedding-3-small (1536 dims)
   Indexed at    : 2026-06-01T21:33:55.398259


---
## 📖 Referencia rápida del sistema

### Claves Redis generadas por el sistema
```
normativa:index          → SET con todos los MD5 de chunks indexados
normativa:doc:<md5>      → JSON: document_name, document_type, article,
                                  section, content, document_uri, criticality,
                                  topic, source_owner, updated_at, embed_model
normativa:emb:<md5>      → bytes float32 del vector (1536 dims)
```

### Comandos de mantenimiento del índice
```python
ver_estado_redis()              # estado actual del índice
indexar_normativa()             # indexa solo si normativa:index está vacío
indexar_normativa(forzar=True)  # re-indexa aunque ya existan datos
REDIS_CLIENT.delete('normativa:index')  # limpiar índice (requiere re-indexar)
```

### Veredictos posibles
| Veredicto | Condición |
|---|---|
| `OK` | Evidencia suficiente, sin PII expuesta, recomendaciones coherentes |
| `ISSUES` | evidence_gap, weak_citation, PII detectada o inconsistencia |
| `CORRECTED` | Errores menores corregidos directamente por el fiscalizador |
| `INCOMPLETE` | Consulta sin antecedentes técnicos suficientes |